## Binary Classification

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("CarPrice.csv")

# Encode all object (categorical) columns with LabelEncoder
cat_features = [feature for feature in df.columns if df[feature].dtype == 'object']
encoder = LabelEncoder()

for feature in cat_features:
    df[feature] = encoder.fit_transform(df[feature])

# Let's say we are predicting fueltype (0 for gas, 1 for diesel).
# Make sure "fueltype" is indeed 0/1 if not already:
# df['fueltype'] = df['fueltype'].map({'gas': 0, 'diesel': 1}) 

# Check how many unique CarName values
print("Unique CarName count:", df['CarName'].nunique())

# Let's pick our features: everything from column 1 to second-last,
# then drop 'CarName' if it's not useful as a feature
X = df.iloc[:, 1:-1]  # or define your own subset of columns
X = X.drop('CarName', axis=1, errors='ignore')  # Only drop if CarName is in X

# Now define y as the fueltype for binary classification
# df.columns might differ, so adjust accordingly
y = df['fueltype']

# Scale X
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

# Train/test split
x_train, x_test, y_train, y_test = train_test_split(X_scaled, y, 
                                                    test_size=0.2, 
                                                    random_state=0)
n_samples = x_train.shape[0]
n_features = x_train.shape[1]
print(f'n_samples: {n_samples}, n_features: {n_features}')

# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).unsqueeze(1)

# Convert features to PyTorch tensors
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

# Define the PyTorch model
class BinaryClassificationModel(nn.Module):
    def __init__(self, n_features):
        super(BinaryClassificationModel, self).__init__()
        self.fc1 = nn.Linear(n_features, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 32)
        self.fc5 = nn.Linear(32, 8)
        self.fc6 = nn.Linear(8, 8)
        self.fc7 = nn.Linear(8, 1)  # Single output for binary classification

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # Apply ReLU activation
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = torch.relu(self.fc6(x))
        x = torch.sigmoid(self.fc7(x))  # Sigmoid activation for binary classification
        return x


In [ ]:

# Define model, loss, and optimizer
n_features = x_train.shape[1]  # Number of input features
model = BinaryClassificationModel(n_features)

criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)


# Create a dataset and split into training and validation sets
dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=10)


In [ ]:

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            y_pred = model(x_val)
            loss = criterion(y_pred, y_val)
            val_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss / len(train_loader):.4f}, Validation Loss: {val_loss / len(val_loader):.4f}")

# Evaluation on the test set
model.eval()
with torch.no_grad():
    y_pred_test = model(x_test_tensor)
    test_loss = criterion(y_pred_test, y_test_tensor).item()
    y_pred_classes = (y_pred_test >= 0.5).float()
    accuracy = (y_pred_classes == y_test_tensor).float().mean().item()

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


# Multiclass Classification

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1) Load Data
df = pd.read_csv("CarPrice.csv")

# 2) Encode the target column ('carbody')
df['carbody'] = df['carbody'].astype('category')
df['carbody_code'] = df['carbody'].cat.codes  # Convert categories to integer codes
y = df['carbody_code'].to_numpy()  # Convert to NumPy array (shape: [num_samples])

# 3) Choose Features
features = [
    'fueltype', 'aspiration', 'wheelbase', 'enginesize', 'horsepower',
    'citympg', 'highwaympg'
]
X = df[features]


In [ ]:

# Encode categorical columns in X
for col in ['fueltype', 'aspiration']:
    if X[col].dtype == object:
        X[col] = LabelEncoder().fit_transform(X[col])

# Scale features (optional, depending on data distribution)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 4) Train/Test Split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # Use long for classification
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create datasets and dataloaders
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10)


In [ ]:

# 5) Define Model
class MulticlassClassificationModel(nn.Module):
    def __init__(self, n_features, n_classes):
        super(MulticlassClassificationModel, self).__init__()
        self.fc1 = nn.Linear(n_features, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 32)
        self.fc5 = nn.Linear(32, 8)
        self.fc6 = nn.Linear(8, 8)
        self.fc7 = nn.Linear(8, n_classes)  # Final layer with n_classes output

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = torch.relu(self.fc6(x))
        x = torch.softmax(self.fc7(x), dim=1)  # Softmax for multi-class output
        return x


In [ ]:

n_features = x_train.shape[1]
n_classes = len(df['carbody'].unique())  # Number of classes
model = MulticlassClassificationModel(n_features, n_classes)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()  # 
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss / len(train_loader):.4f}")


In [ ]:

# Evaluation on test set
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        test_loss += loss.item()
        predicted = torch.argmax(y_pred, dim=1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

print(f"Test Loss: {test_loss / len(test_loader):.4f}")
print(f"Test Accuracy: {correct / total:.4f}")
